In [1]:
! pip install langchain
! pip install langchain-neo4j langchain-groq

In [49]:
import os
import unicodedata
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

load_dotenv()

# 1. Conectar o LangChain ao seu Neo4j local
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# 2. Inicializar o "Cérebro"
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0 
)

# 3. O Prompt Turbinado com APOC e Regras de Negócio
cypher_template = """Você é um especialista em banco de dados Neo4j traduzindo perguntas em português para queries Cypher.

REGRAS OBRIGATÓRIAS:
1. Use APENAS os nós e relacionamentos fornecidos no schema.
2. NUNCA faça buscas textuais exatas usando '='. 
3. SEMPRE use a função apoc.text.clean() EM AMBOS OS LADOS da comparação com CONTAINS. O APOC remove acentos e espaços.
   Exemplo exigido: apoc.text.clean(e.nome) CONTAINS apoc.text.clean('itau cartoes')
4. CONTEXTO DE NEGÓCIO DA BASE DE DADOS:
   - Marcas, bancos, empresas e aplicativos (ex: Itaú Cartões, Uber, iFood, AmoraDev) ficam SEMPRE no nó Estabelecimento (e.nome).
   - Setores amplos, tipos de despesas e receitas (ex: Transporte, Alimentação, Juros, Salário) ficam SEMPRE no nó Categoria (cat.nome).
   - Transações de saída de dinheiro têm a propriedade t.tipo = 'Saída'. Transações de ganho de dinheiro têm t.tipo = 'Entrada'.
5. Retorne APENAS o código Cypher válido, sem explicações.

Schema:
{schema}

Pergunta do usuário: {question}
Query Cypher:"""

cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template=cypher_template
)

# 4. Criar o Agente
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    cypher_prompt=cypher_prompt,
    verbose=True, 
    allow_dangerous_requests=True 
)

# 5. O Escudo Python
def limpar_texto(texto):
    texto_sem_acento = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto_sem_acento.lower()

# 6. O Sistema de Memória e o Loop do Chat
historico_chat = []
limite_memoria = 3 # Lembra apenas das últimas 3 perguntas para economizar tokens

print("Agente Financeiro Online! (Digite 'sair' para encerrar)\n" + "-"*50)

while True:
    # Captura a pergunta do usuário no terminal
    pergunta_crua = input("Você: ")
    
    if pergunta_crua.lower() in ['sair', 'exit', 'quit']:
        print("Encerrando o chat. Até a próxima!")
        break
        
    pergunta_limpa = limpar_texto(pergunta_crua)
    
    # Injeta a memória na pergunta se houver histórico
    if historico_chat:
        # Formata o histórico em texto
        texto_historico = "\n".join([f"Usuário perguntou: {h['user']}\nIA respondeu: {h['ia']}" for h in historico_chat])
        # Constrói o super-prompt
        pergunta_com_contexto = f"Baseado neste histórico de conversa:\n{texto_historico}\n\nResponda à nova pergunta: {pergunta_limpa}"
    else:
        pergunta_com_contexto = pergunta_limpa

    # Invoca o Agente (passando a pergunta turbinada com o histórico)
    print("Processando no Neo4j...")
    resposta = chain.invoke({"query": pergunta_com_contexto})
    resultado_ia = resposta['result']
    
    print(f"IA: {resultado_ia}\n")
    
    # Salva a interação atual na memória e limpa os mais antigos
    historico_chat.append({"user": pergunta_limpa, "ia": resultado_ia})
    if len(historico_chat) > limite_memoria:
        historico_chat.pop(0)

Agente Financeiro Online! (Digite 'sair' para encerrar)
--------------------------------------------------
Processando no Neo4j...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)-[:PERTENCE_A_CATEGORIA]->(cat:Categoria)
WHERE apoc.text.clean(cat.nome) CONTAINS apoc.text.clean('salario')
  AND t.tipo = 'Entrada'
RETURN sum(t.valor) AS salario;
Full Context:
[{'salario': 3000.0}]

> Finished chain.
IA: O valor do salário é 3000.0.

Processando no Neo4j...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)-[:NO_ESTABELECIMENTO]->(e:Estabelecimento)
WHERE t.tipo = 'Saída'
  AND apoc.text.clean(e.nome) CONTAINS apoc.text.clean('uber')
RETURN sum(t.valor) AS gastoUber;
Full Context:
[{'gastoUber': 132.3}]

> Finished chain.
IA: O gasto com Uber foi 132.3.

Processando no Neo4j...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:T